# Homework Pyspark SQL

Note: This notebook will contain only eligible practical questions.

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

In [2]:
# Initialize SparkSession
spark = SparkSession.builder \
    .master("local[*]") \
    .appName("pyspark_sql") \
    .getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/06/19 19:54:44 WARN Utils: Your hostname, JACK2000-MAC.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.158 instead (on interface en0)
26/06/19 19:54:44 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/06/19 19:54:45 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [ ]:
# Load the data
df = spark.read.parquet('../data/raw/yellow/2025/yellow_tripdata_2025-11.parquet')

In [13]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True), StructField('cbd_congestio

# Q3

In [7]:
# Python way: Yellow schema uses tpep_pickup_datetime
df.filter(F.to_date("tpep_pickup_datetime") == "2025-11-15").count()

162604

In [15]:
# SQL way
df.createOrReplaceTempView("trips")
spark.sql("""
  select count(*) as count
  from trips
  where date(tpep_pickup_datetime) = '2025-11-15'
""").show()

+------+
| count|
+------+
|162604|
+------+



## Q4

In [ ]:
# Python query
dur = (df.withColumn(
  "duration_hours",
  (F.col("tpep_dropoff_datetime").cast("long")
   - F.col("tpep_pickup_datetime").cast("long"))
   / 3600.0
))

dur.select(F.max("duration_hours").alias("max_hours")).show()

# Casting a timestamp to long gives epoch seconds; subtract and divide by 3600 for hours

In [17]:
# SQL query
spark.sql("""
  select
      max((unix_timestamp(tpep_dropoff_datetime) -
      unix_timestamp(tpep_pickup_datetime)) / 3600.0)
      as max_hours
  from trips
""").show()

+---------+
|max_hours|
+---------+
|90.646667|
+---------+



# Q6

In [19]:
# Download the data
!curl -o ../data/raw/zone/taxi_zone_lookup.csv https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 12331  100 12331    0     0   3  0      0 --:--:-- --:--:-- --:--:--     048k      0 --:--:-- --:--:-- --:--:--  354k


In [21]:
# Python query
zones = (spark.read.option("header", True)
         .csv("../data/raw/zone/taxi_zone_lookup.csv"))
# LocationID, Borough, Zone, service_zone

(df.join(zones, df.PULocationID == zones.LocationID, "left")
  .groupBy("Zone")
  .agg(F.count("*").alias("trips"))
  .orderBy("trips")
  .show(5, truncate=False))

+---------------------------------------------+-----+
|Zone                                         |trips|
+---------------------------------------------+-----+
|Governor's Island/Ellis Island/Liberty Island|1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Arden Heights                                |1    |
|Port Richmond                                |3    |
|Rikers Island                                |4    |
+---------------------------------------------+-----+
only showing top 5 rows


In [22]:
# SQL query
zones.createOrReplaceTempView("zones")
spark.sql("""
  select z.Zone, count(*) as trips
  from trips t
  left join zones z on t.PULocationID = z.LocationID
  group by z.Zone
  order by trips asc
  limit 5
""").show(truncate=False)

+---------------------------------------------+-----+
|Zone                                         |trips|
+---------------------------------------------+-----+
|Governor's Island/Ellis Island/Liberty Island|1    |
|Eltingville/Annadale/Prince's Bay            |1    |
|Arden Heights                                |1    |
|Port Richmond                                |3    |
|Rikers Island                                |4    |
+---------------------------------------------+-----+

